# Сводный анализ и визуализация данных в Pandas/Matplotlib/seaborn/plotly


Разведочный или исследовательский анализ данных (Exploratory Data Analysis, __EDA__) — [это этап работы с данными](https://habr.com/ru/companies/yandex_praktikum/articles/952084/), на котором аналитик изучает доступную информацию перед построением сложных  моделей искусственного интеллекта.  [EDA](https://github.com/ivtipm/ML) включает анализ основных свойств данных, нахождение в них общих закономерностей, распределений и аномалий, постановку начальных гипотез, зачастую с использованием инструментов визуализации.

[Цели разведочного анализа данных](https://habr.com/ru/companies/otus/articles/752434/)

* __Понимание структуры и характеристик набора данных__
На этом этапе EDA мы пристально изучаем структуру и характеристики данных, чтобы полностью понять, с чем имеем дело. Это включает в себя обзор размера набора данных, типов переменных, наличия пропущенных значений, дубликатов и других важных аспектов. Разведка на этом уровне позволяет нам сформировать четкое представление о данных, что становится фундаментом для дальнейшего анализа.

* __Выявление редких или нестандартных значений (аномалий и выбросов)__
Иногда данные могут хранить в себе сюрпризы, которые могут существенно повлиять на наши выводы. Используя методы EDA мы обнаруживаем аномалии и выбросы в наших данных. Эти значения, отклоняющиеся от общего паттерна, могут возникать из-за ошибок ввода, случайных событий или даже указывать на систематические проблемы в сборе данных. Определение и устранение таких аномалий становится критической частью нашего анализа, чтобы гарантировать надежность результатов.

* __Идентификация связей (корреляций) между переменными__
Используя статистические меры EDA позволяет идентифицировать взаимосвязи между переменными, что помогает понять, как одни факторы влияют на другие. Это может стать отправной точкой для формулирования гипотез и принятия важных бизнес-решений.

* __Подготовка данных для дальнейших этапов анализа__
На этом этапе EDA мы обрабатываем данные для того, чтобы подготовить их к более сложным аналитическим методам. Мы чистим данные от шума, заполняем пропущенные значения, проводим масштабирование или преобразования переменных, чтобы обеспечить их качественную и интерпретируемую структуру. Только с надлежаще подготовленными данными мы можем достичь точных и надежных результатов на последующих этапах анализа.



# Import библиотек

In [ ]:
import pandas as pd
import random
import numpy as np

Магическая команда, которая включает встроенное отображение графиков прямо в ячейках ноутбука.

In [ ]:
%matplotlib inline

In [ ]:
import plotly
import seaborn as sns
import matplotlib.pyplot as plt

## Загрузка данных 

Следует загрузить свой вариант датасета, полученного после прохождения предыдущей практики. Но для примера загрузим наш датасет

In [ ]:
df=pd.read_csv('cars_processed.csv')

# EDA - Разведывательный анализ данных.

Цель разведывательного анализа данных - попытка предварительного понимания того, с чем вы имеете дело. Например, результатами анализа могут быть удаление аномалий, заполнение (удаление) пропусков, визуализация данных, выделение признаков или получение (или преобразование) новых признаков, удаление некоторых шумов, помех или других нерегулярных значений.


In [ ]:
df.info()

2. Вспомним про то, как проверить данные на количество пропусков и дубликатов

In [ ]:
df.isna().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates()

## Сводный анализ признаков 

Во многих случаях аналитики должны предоставить агрегированные или сгруппированные данные. Эта информация помогает понять общие характеристики в определенных разрезах данных. Часто этого достаточно, чтобы достичь бизнес-цели или принимать решения.

Также сводный анализ позволяет достаточно просто нати наиболее редкие записи данных или наиболее редкие комбинации данные

### Описательные статистики

Попробуем проанализировать данные 

1. посмотрим на то, какие у нас есть категориальные и численные столбцы.
   
[Напомним, что категориальные переменные](https://education.yandex.ru/handbook/data-analysis/article/rabota-s-kategorialnymi-dannymi-v-python) — это те переменные, которые выражены ограниченным диапазоном значений. Например, группа крови или семейное положение.



In [ ]:
num_columns = df.select_dtypes(include='number').columns
cat_columns = df.select_dtypes(include= ['object', 'bool']).columns

Статистика по числовым данным

In [ ]:
df.describe()

Из анализа видно, что, например, 
* Наиболее частый год обращения автомобией 2009, макимальный год 2021.
* Однако, большая часть автомобилей укладывается в диапазон от 2004 до 2014 годов (2009 $\pm$5 лет).
* Вероятно нас не интересуют автомобили сильно старше 2004 года. Например возьмем 1.5 размаха по 5 лет (1996 - 2021).

Аналогично примеру выше посчитаем статистик по категориальным признакам. Это будут признаки с указанием типов данных `object` и `bool`

In [ ]:
df.describe(include=['object', 'bool'])

Другим важным инструментом анализа является группировка данных. 

В общем случае группировка данных в Pandas выглядит следующим образом:
```python
df.groupby(by=grouping_columns)[columns_to_show].function()
```

1. К датафрейму применяется метод `groupby`, который групперует данные по `grouping_columns` – признаку или набору признаков.
2. Выбираем нужные нам столбцы (`columns_to_show`).
3. К полученным группам применяется функция или несколько функций.

Попробуем сгруппировать данные по производителю ('Make'). Посмотрим сколько автомобилей кажого производителя имеется по годам. В качестве функции группировки будем использовать подсчет значений в группе (`count`), минимальный и максимальный год автомобиля в группе `np.min, np.max`. Для того, чтобы использовать сразу 3 функции вместе воспользуемся методом `agg`

In [ ]:
df.groupby(['Model','Year'])[['Price']].mean()

### Упражнение

1.Вывести 10 самых дорогих марок автомобилей в нашей базе данных

In [ ]:
df.groupby(['Make'])['Price'].mean().sort_values(ascending=False).head(10)